# NB08 — The other half of the dissociation (optional)

**Plan §9. Budget: ~3 h. Needs the same single-GPU 80 GB pod as NB03. Run only if NB00–NB07
are done.**

Rotation kills basis while holding behavior fixed. The dual would kill behavior while holding
basis fixed.

**This experiment is not that dual, and it does not bracket the question.** Input ablation
passes no activation, so alignment is *inert* on that task — it is not a variable that can take
the value 0.85 there. The rotation point (behavioral divergence 0, alignment 0, patching task)
and the base/instruct point (behavioral divergence > 0, patching-task alignment ≈ 0.85, but
measured on input ablation) do not sit on the same surface and cannot be compared. The genuine
dual requires base/instruct on **patching**, which means regenerating every ground-truth
patching label for the base model — the exact cost NB02 exists to avoid.

What this experiment *is* worth ~3 hours for, as its own claim: **isolating the contribution of
behavioral similarity on a task where basis compatibility cannot operate by construction.**

There is a live thread there. The paper's Table 2 shows the Qwen3 self-margin is *larger* on
input ablation (83.4 vs 58.1 exact match) than on patching (64.0 vs 54.1) — the task with no
internals shows the bigger effect. That is already awkward for the privileged-access reading
and nobody has followed it up.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Two 8B explainers trained on the ablation task, one after the other.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


## 0. Model availability

Qwen3 ships instruction-tuned checkpoints under the plain name; the pretrained-only
checkpoints carry a `-Base` suffix and are not published for every size. Check before
planning around them, and fall back to a size where both exist rather than silently
comparing two instruct models.


In [ ]:
from huggingface_hub import model_info

CANDIDATES = [("Qwen/Qwen3-8B-Base", "Qwen/Qwen3-8B"),
              ("Qwen/Qwen3-4B-Base", "Qwen/Qwen3-4B"),
              ("Qwen/Qwen3-1.7B-Base", "Qwen/Qwen3-1.7B")]

PAIR = None
for base_id, instruct_id in CANDIDATES:
    try:
        model_info(base_id)
        model_info(instruct_id)
        PAIR = (base_id, instruct_id)
        print(f"using {base_id} / {instruct_id}")
        break
    except Exception as err:                       # noqa: BLE001
        print(f"unavailable: {base_id} ({type(err).__name__})")

assert PAIR, "no base/instruct pair available — this experiment cannot run as specified"
BASE_ID, INSTRUCT_ID = PAIR


## 1. Relabel the task for each target

This is the step that makes the experiment cheap, and the reason §9 chose input ablation:
regenerating MMLU responses is what `iteration.ipynb` already does. Each target model answers
the hinted and unhinted prompts itself, and those answers become the ground truth its explainer
must predict.

The dataset's shipped labels are Qwen3-8B's, so they cannot be reused for a different target
— using them would be comparing explainers of a model that isn't in the experiment.


In [ ]:
import json

import torch
from transformers import AutoModelForCausalLM

import se_common as S

N_RELABEL = 2048 + C.EVAL_SIZE
tokenizer = S.load_tokenizer(INSTRUCT_ID)
abl = S.build_ablation_dataset(seed=C.SEED)


@torch.no_grad()
def answer_batch(model, prompts, batch_size=16):
    """Greedy single-letter answers, in the format the ablation task expects."""
    out = []
    tokenizer.padding_side = "left"
    for start in range(0, len(prompts), batch_size):
        msgs = [[{"role": "system", "content": S.ABLATION_SYSTEM_PROMPT},
                 {"role": "user", "content": p}] for p in prompts[start:start + batch_size]]
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True,
                                               enable_thinking=False) for m in msgs]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=8, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for row in tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:],
                                          skip_special_tokens=True):
            letter = next((ch for ch in row.upper() if ch in "ABCD"), "A")
            out.append(letter)
    return out


def relabel_for(model_id):
    """{index: (answer_without_hint, answer_with_hint)} as produced by `model_id` itself."""
    cache = f"{C.REPORTS_DIR}/labels_{model_id.split('/')[-1]}.json"
    if os.path.exists(cache):
        return {int(k): tuple(v) for k, v in json.load(open(cache)).items()}

    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=S.compute_dtype(), device_map="auto").eval()
    rows = abl.select(range(N_RELABEL))
    no_hint = answer_batch(model, [f"Question: {q}" for q in rows["q"]])
    with_hint = answer_batch(model, [f"Question: {q}\nHint: {h}"
                                     for q, h in zip(rows["q"], rows["hint"])])
    labels = {i: (a, b) for i, (a, b) in enumerate(zip(no_hint, with_hint))}

    json.dump({str(k): list(v) for k, v in labels.items()}, open(cache, "w"))
    del model
    torch.cuda.empty_cache()
    return labels


labels = {mid: relabel_for(mid) for mid in (BASE_ID, INSTRUCT_ID)}
for mid, lab in labels.items():
    changed = sum(a != b for a, b in lab.values())
    print(f"{mid}: {changed}/{len(lab)} = {changed/len(lab):.3f} of items change under the hint")


In [ ]:
# behavioral divergence between the two targets — the axis this experiment varies
shared = set(labels[BASE_ID]) & set(labels[INSTRUCT_ID])
agree_no_hint = sum(labels[BASE_ID][i][0] == labels[INSTRUCT_ID][i][0] for i in shared)
agree_verdict = sum((labels[BASE_ID][i][0] != labels[BASE_ID][i][1])
                    == (labels[INSTRUCT_ID][i][0] != labels[INSTRUCT_ID][i][1])
                    for i in shared)
print(f"answer agreement between targets  : {agree_no_hint/len(shared):.3f}")
print(f"has-changed agreement             : {agree_verdict/len(shared):.3f}")
print("\nThese two models share a frame almost exactly but differ behaviorally — the")
print("mirror image of the rotation arm, which holds behavior fixed and destroys the frame.")


## 2. The 2×2: each model explains each target

**One tokenizer, both explainers.** Qwen3 base and instruct checkpoints share a vocabulary,
but the `-Base` tokenizer has no chat template, and `assistant_only_loss` needs one. Both arms
therefore use the instruct tokenizer and its template.

State the caveat rather than hiding it: the base model is being asked to work in a chat format
it was never trained on, so "behavioral divergence" here is partly format familiarity. It
biases *against* the base model symmetrically in both the self and cross cells, so the
self-margin comparison survives, but the absolute scores for base-as-explainer are a floor,
not an estimate.


In [ ]:
import pandas as pd

N_TRAIN = 2048
results = []

for target_id in (BASE_ID, INSTRUCT_ID):
    ds = S.build_ablation_dataset(seed=C.SEED, labels=labels[target_id])
    for explainer_id in (BASE_ID, INSTRUCT_ID):
        tag = (f"expl_{explainer_id.split('/')[-1]}__target_{target_id.split('/')[-1]}")
        print(f"\n=== {tag} " + "=" * 30)
        S.run_ablation_training(N_TRAIN, tag, tokenizer, ds, explainer_model_id=explainer_id,
                                task="ablation_dissociation")
        scores = S.eval_ablation(N_TRAIN, tag, tokenizer, ds, explainer_model_id=explainer_id,
                                 task="ablation_dissociation", keep_generations=False)
        scores.update({"explainer": explainer_id, "target": target_id,
                       "is_self": explainer_id == target_id})
        results.append(scores)
        print(f"  exact_match {scores['exact_match']:.3f} | "
              f"content_match {scores['content_match']:.3f}")

diss = pd.DataFrame(results)
diss.to_csv(f"{C.REPORTS_DIR}/dissociation.csv", index=False)
diss[["explainer", "target", "is_self", "exact_match", "has_changed_f1", "content_match"]]


In [ ]:
metric = "exact_match"
self_scores = diss[diss.is_self][metric].mean()
cross_scores = diss[~diss.is_self][metric].mean()

print("BASE/INSTRUCT DISSOCIATION")
print("=" * 70)
print(diss.pivot_table(index="explainer", columns="target", values=metric).round(3).to_string())
print(f"\nmean self  : {self_scores:.3f}")
print(f"mean cross : {cross_scores:.3f}")
print(f"self margin: {self_scores - cross_scores:+.3f}")
print()
print("Reading it, given no activation is passed in this task at all:")
print("  margin ~ 0  -> the self-advantage on input ablation was never about self-knowledge;")
print("                 two models with the same frame explain each other as well as themselves")
print("  margin > 0  -> something survives that is neither basis nor architecture: behavioral")
print("                 self-similarity is doing work, and the rotation result needs that caveat")


## 3. Putting the two halves together


In [ ]:
rot = json.load(open(f"{C.REPORTS_DIR}/verdicts.json"))

print("THE DECOMPOSITION")
print("=" * 70)
print(f"{'experiment':<28} {'behavioral div.':>16} {'alignment':>12} {'self margin':>13}")
print(f"{'rotation (NB03)':<28} {'0':>16} {'0':>12} "
      f"{max(rot['gaps'].values(), default=float('nan')):>13.3f}")
print(f"{'base/instruct (NB08)':<28} {'> 0':>16} {'~0.85':>12} "
      f"{self_scores - cross_scores:>13.3f}")
print()
print("If explainer score tracks alignment in the first and behavior in the second, the two")
print("contributions separate. If both margins are near zero, neither privileged access nor")
print("basis compatibility is carrying the effect and the remaining explanation is task")
print("structure — which would be a finding in its own right, and a bigger one.")
